# GraphForge — Rejection-Sampling SFT on a free Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nithin062006/scaler/blob/main/training/notebook.ipynb)

This notebook runs the full pipeline end-to-end against the GraphForge OpenEnv environment:

1. Clone the repo and install deps (1 min on T4)
2. Baseline-eval Qwen2.5-0.5B-Instruct against the tier-0 task
3. Generate trajectories (oracle + live model) and reject-sample
4. SFT the kept trajectories (TRL SFTTrainer + LoRA)
5. Trained-eval the same model and write all hackathon plots

Expected wall-clock: ~10–20 min on a T4.

## 1. Setup

In [ ]:
import os, subprocess, pathlib

REPO_URL = 'https://github.com/nithin062006/scaler.git'
cwd = pathlib.Path(os.getcwd())

# Idempotent: handles fresh runtime, restarted runtime, and re-runs.
if (cwd / 'graphforge').exists() and (cwd / 'env').exists():
    print(f'Already inside repo: {cwd}')
elif (cwd / 'graphforge_repo').exists():
    os.chdir('graphforge_repo')
    print(f'Cd-ed into existing clone: {os.getcwd()}')
else:
    subprocess.check_call(['git', 'clone', '-q', REPO_URL, 'graphforge_repo'])
    os.chdir('graphforge_repo')
    print(f'Cloned + cd-ed: {os.getcwd()}')

print(os.listdir('.'))

In [ ]:
# Install runtime + training deps. peft & trl handle the SFT side.
%pip install -q -e ".[training]"
%pip install -q peft
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Patch HfPolicy.sample for newer transformers' apply_chat_template return type.
# Idempotent — overwrites the file on every run. Safe to keep even after the
# fix lands upstream (writing the same content is a no-op observationally).
import pathlib, sys

_FIXED = '''"""Policy interface and stub policies."""

from __future__ import annotations
from typing import Iterator, Protocol, runtime_checkable
from graphforge.training.prompt import Message


@runtime_checkable
class Policy(Protocol):
    def sample(self, messages: list[Message]) -> str: ...


class ScriptedPolicy:
    def __init__(self, completions):
        self._iter = iter(completions)
    def sample(self, _messages):
        return next(self._iter)


class HfPolicy:
    def __init__(self, model, tokenizer, *, max_new_tokens=384, temperature=0.7, top_p=0.95):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p

    def sample(self, messages):
        import torch
        tok = self.tokenizer
        text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = tok(text, return_tensors="pt")
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=self.temperature,
                top_p=self.top_p,
                pad_token_id=tok.eos_token_id,
            )
        prompt_len = inputs["input_ids"].shape[-1]
        gen = out_ids[0, prompt_len:]
        return tok.decode(gen, skip_special_tokens=True)
'''
pathlib.Path('graphforge/training/policy.py').write_text(_FIXED)

# Drop any cached graphforge imports so the next cell re-imports fresh.
_dropped = [k for k in list(sys.modules) if k.startswith('graphforge')]
for k in _dropped:
    del sys.modules[k]
print(f"Patched policy.py; cleared {len(_dropped)} cached imports.")

In [ ]:
# Patch train.py for TRL 0.12+ API drift:
#  - drop `max_seq_length` and `dataset_text_field` kwargs (removed in 0.12+)
#  - add SFTTrainer(processing_class=...) fallback for transformers 4.46+
import pathlib, re, sys

p = pathlib.Path('training/train.py')
src = p.read_text()
lines = src.split('\n')
filtered = [
    l for l in lines
    if not re.search(r'\bmax_seq_length\s*=', l)
    and not re.search(r'\bdataset_text_field\s*=', l)
]
removed = len(lines) - len(filtered)
text = '\n'.join(filtered)

if 'processing_class=tok' not in text:
    old = '''    trainer = SFTTrainer(
        model=model,
        args=sft_cfg,
        train_dataset=ds,
        tokenizer=tok,
    )'''
    new = '''    try:
        trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=ds, processing_class=tok)
    except TypeError:
        trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=ds, tokenizer=tok)'''
    text = text.replace(old, new)

p.write_text(text)
print(f"Patched train.py: removed {removed} arg lines, added SFTTrainer fallback.")

# Drop cached imports so changes take effect.
_dropped = [k for k in list(sys.modules) if k.startswith('graphforge') or k.startswith('training')]
for k in _dropped:
    del sys.modules[k]
print(f"Cleared {len(_dropped)} cached imports.")

In [ ]:
from pathlib import Path
from training.config import TrainConfig
from training.train import run

# Tight Colab-T4-friendly config. Total wall-clock ~5–8 min:
#   baseline eval:   3 ep × 8 turns × 64 tok ≈ 1.5 min
#   oracle gen:      20 scripted, instant
#   model explore:   2 ep × 8 turns × 64 tok ≈ 1 min
#   SFT 1 epoch:     ~80 examples on LoRA, ~2 min
#   trained eval:    3 ep × ~4 turns × 64 tok ≈ 1 min
cfg = TrainConfig(
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    task_id='t0.email_validator',
    max_new_tokens=64,
    episode_cap=8,
    n_oracle=20,
    n_explore=2,
    reward_threshold=5.0,
    epochs=1,
    learning_rate=2e-4,
    batch_size=1,
    gradient_accumulation_steps=2,
    use_lora=True,
    n_eval_episodes=3,
    out_dir=Path('outputs'),
    plots_dir=Path('plots'),
)
summary = run(cfg)
print('=' * 60)
print(f"baseline mean = {summary['baseline_eval']['mean_reward']:+.2f}")
print(f"trained mean  = {summary['trained_eval']['mean_reward']:+.2f}")
print('=' * 60)

## 2. Run the full pipeline

`training.train.run` does baseline eval → trajectory generation → SFT → trained eval → plots, all in one call.

In [ ]:
import json, statistics
from pathlib import Path
from IPython.display import Image, display, Markdown

b = json.loads(Path('outputs/baseline_eval.json').read_text())
t = json.loads(Path('outputs/trained_eval.json').read_text())

def stats(name, e):
    rs = e['rewards']
    return {
        'name': name, 'n': len(rs),
        'mean': statistics.fmean(rs),
        'std': statistics.pstdev(rs) if len(rs) > 1 else 0.0,
        'min': min(rs), 'max': max(rs),
        'completion_rate': e['completion_rate'],
        'parse_fail_avg': statistics.fmean(e['parse_failure_rates']),
        'rewards': rs,
    }

bs, ts = stats('Baseline', b), stats('Trained', t)
delta = ts['mean'] - bs['mean']

print('=' * 72)
print(f"{'metric':<28} {'baseline':>14} {'trained':>14} {'Δ':>14}")
print('-' * 72)
def row(label, key, fmt='+.2f'):
    bv, tv = bs[key], ts[key]
    print(f"{label:<28} {bv:>14{fmt}} {tv:>14{fmt}} {tv - bv:>+14{fmt}}")
row('mean terminal reward', 'mean')
row('std', 'std')
row('min', 'min')
row('max', 'max')
row('completion rate', 'completion_rate', '.0%')
row('avg parse-fail rate/ep', 'parse_fail_avg', '.0%')
print('-' * 72)
print(f"absolute Δ mean reward = {delta:+.2f}")
print('=' * 72)

print(f"\nBaseline rewards: {[round(r, 2) for r in bs['rewards']]}")
print(f"Trained rewards : {[round(r, 2) for r in ts['rewards']]}")

if delta > 5:
    verdict = f"✅ Training made the model substantially better (Δ = {delta:+.2f})."
elif delta > 0:
    verdict = f"⚠️ Training improved the model but the margin is small (Δ = {delta:+.2f})."
else:
    verdict = f"❌ Training did not improve the model (Δ = {delta:+.2f})."
print(f"\n{verdict}")

display(Markdown("\n## Plots"))
for name in ['comparison.png', 'baseline_rewards.png', 'trained_rewards.png',
             'baseline_hist.png', 'trained_hist.png', 'loss_curve.png']:
    p = Path('plots') / name
    if p.exists():
        display(Markdown(f"**{name}**"))
        display(Image(str(p)))
    else:
        print(f"(missing: {name})")

## 3. Show the plots

In [ ]:
from IPython.display import Image, display
for name in ['comparison.png', 'baseline_rewards.png', 'trained_rewards.png', 'loss_curve.png']:
    p = Path('plots') / name
    if p.exists():
        print(name)
        display(Image(str(p)))

## 4. Commit the plots back to the repo

Once you're happy with the run, copy `plots/*.png` and `outputs/summary.json` into your fork and push. The README embeds them automatically.